<a href="https://colab.research.google.com/github/Kongbeng-21/SuperAi/blob/main/thai_captioning_nllb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Thai Language Image Captioning
**Strategy:** BLIP-large (EN caption) → NLLB-200 (EN→TH) → submit

- ต้องเปิด **GPU T4** และ **Internet** ใน Kaggle settings

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
!pip install -q transformers==4.40.0 accelerate sentencepiece

## 1. Load Data

In [ ]:
import pandas as pd
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input/competitions/super-ai-engineer-ss-6-thai-language-image-captioning')
test_dir   = INPUT_ROOT / 'test' / 'test'
sample_sub = pd.read_csv(INPUT_ROOT / 'sample_submission.csv')

test_images = sorted(test_dir.glob('*.jpg'))
id2path = {int(p.stem): str(p) for p in test_images}

print('Test images:', len(test_images))
print('Sample sub rows:', len(sample_sub))
sample_sub.head(3)

## 2. Load BLIP-large (Caption Generator)

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

BLIP_MODEL = 'Salesforce/blip-image-captioning-large'
print(f'Loading {BLIP_MODEL}...')

blip_processor = BlipProcessor.from_pretrained(BLIP_MODEL)
blip_model = BlipForConditionalGeneration.from_pretrained(
    BLIP_MODEL,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32
).to(DEVICE)
blip_model.eval()
print('BLIP loaded ✓')

## 3. Load NLLB-200 (EN → TH Translation)

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# NLLB-200 distilled 600M — public, no login required, แปลไทยดี
NLLB_MODEL = 'facebook/nllb-200-distilled-600M'
print(f'Loading {NLLB_MODEL}...')

nllb_tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(
    NLLB_MODEL,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32
).to(DEVICE)
nllb_model.eval()
print('NLLB loaded ✓')

# NLLB ใช้ language code แบบนี้
SRC_LANG = 'eng_Latn'  # English
TGT_LANG = 'tha_Thai'  # Thai

## 4. Generate EN Captions with BLIP

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm

BLIP_BATCH = 8  # ลดถ้า OOM

class TestImageDataset(Dataset):
    def __init__(self, image_ids, id2path, processor):
        self.image_ids = image_ids
        self.id2path   = id2path
        self.processor = processor

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        path   = self.id2path.get(img_id)
        img    = Image.open(path).convert('RGB') if path else Image.new('RGB', (384,384), (128,128,128))
        pixel_values = self.processor(images=img, return_tensors='pt')['pixel_values'].squeeze(0)
        return pixel_values, img_id


image_ids = sample_sub['image_id'].tolist()
dataset   = TestImageDataset(image_ids, id2path, blip_processor)
loader    = DataLoader(dataset, batch_size=BLIP_BATCH, shuffle=False, num_workers=2, pin_memory=True)

en_captions = []
dtype = torch.float16 if DEVICE == 'cuda' else torch.float32

with torch.no_grad():
    for pixel_values, _ in tqdm(loader, desc='BLIP generating'):
        pixel_values = pixel_values.to(DEVICE, dtype=dtype)
        out = blip_model.generate(
            pixel_values=pixel_values,
            max_new_tokens=60,
            num_beams=5,
            length_penalty=1.2,
            repetition_penalty=1.3,
        )
        en_captions.extend(blip_processor.batch_decode(out, skip_special_tokens=True))

print(f'Generated {len(en_captions)} captions')
for c in en_captions[:5]:
    print(' ', c)

## 5. Translate EN → TH with NLLB-200

In [ ]:
MT_BATCH = 32

def translate_nllb(texts, tokenizer, model, src_lang, tgt_lang, device, batch_size=MT_BATCH):
    tokenizer.src_lang = src_lang
    forced_bos = tokenizer.convert_tokens_to_ids(tgt_lang)
    results = []

    for i in tqdm(range(0, len(texts), batch_size), desc='NLLB translating'):
        batch = texts[i:i+batch_size]
        tok = tokenizer(
            batch,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=128
        ).to(device)

        with torch.no_grad():
            out = model.generate(
                **tok,
                forced_bos_token_id=forced_bos,
                max_new_tokens=128,
                num_beams=4,
                length_penalty=1.0,
            )
        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
        results.extend(decoded)

    return results


th_captions = translate_nllb(
    en_captions, nllb_tokenizer, nllb_model,
    SRC_LANG, TGT_LANG, DEVICE
)

print(f'Translated {len(th_captions)} captions')
print('\nExamples:')
for en, th in zip(en_captions[:5], th_captions[:5]):
    print(f'  EN: {en}')
    print(f'  TH: {th}\n')

## 6. Save Submission

In [ ]:
submission = sample_sub.copy()
submission['caption'] = th_captions

assert len(submission) == len(th_captions)
assert submission['caption'].isna().sum() == 0

out_path = '/kaggle/working/submission.csv'
submission.to_csv(out_path, index=False)
print(f'Saved {len(submission)} rows → {out_path}')
submission.head(10)